In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer
from datasets import load_dataset
import torch
import accelerate
import pandas as pd

print(f"Accelerate 版本: {accelerate.__version__}")
print(f"PyTorch 版本: {torch.__version__}")

model_name = "hfl/chinese-roberta-wwm-ext"
cache_dir = "model/hfl_chinese-roberta-wwm-ext"

# 首先检查CSV文件结构
print("检查CSV文件结构...")
df = pd.read_csv("text.csv")
print(f"数据行数: {len(df)}")
print(f"列名: {df.columns.tolist()}")
print("\n标签分布:")
print(df['label'].value_counts())
print("\n数据样例:")
print(df.head())

# 加载模型和分词器
print("\n正在加载模型和分词器...")
tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=cache_dir)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    cache_dir=cache_dir, 
    num_labels=2  # 2个类别：正面和负面
)
print("模型和分词器加载完成！")

# 加载数据集
print("正在加载数据集...")
dataset = load_dataset("csv", data_files="text.csv")

# 数据预处理函数
def preprocess_function(examples):
    # 分词处理
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128,
        return_tensors=None  # 返回普通字典而不是tensors
    )
    
    # 确保标签是整数类型
    tokenized["labels"] = [int(label) for label in examples["label"]]
    
    return tokenized

# 对数据集进行编码
print("预处理数据...")
encoded_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names  # 移除原始列，只保留处理后的列
)

print(f"预处理后的数据集样例:")
print(encoded_dataset["train"][0].keys())

# 设置训练参数
training_args = TrainingArguments(
    output_dir="output/zh_model_1",
    per_device_train_batch_size=4,
    num_train_epochs=5,
    logging_steps=5,
    save_steps=10,
    # learning_rate=2e-5,  # 适合分类任务的学习率
    # warmup_ratio=0.1,    # 使用比例而不是固定步数
    # weight_decay=0.01,
    # evaluation_strategy="no",  # 如果没有验证集
    # save_total_limit=2,
    # logging_dir="./logs",
    # report_to=None,  # 禁用wandb等报告
)

# 创建训练器
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    tokenizer=tokenizer,
)

print("开始训练情感分析模型...")
trainer.train()
print("训练完成！")


Accelerate 版本: 1.10.1
PyTorch 版本: 2.8.0+cu129
检查CSV文件结构...
数据行数: 827
列名: ['text', 'label']

标签分布:
label
困惑    111
惊讶    107
平静    106
难过    104
愤怒    104
开心    102
恐惧     99
厌恶     94
Name: count, dtype: int64

数据样例:
                text label
0  听到这个好消息，我开心得跳了起来！    开心
1  终于完成了这个项目，太有成就感了！    开心
2    和朋友相聚的时光总是这么愉快！    开心
3   看到花开得这么美，心情都变好了！    开心
4   吃到想念已久的美食，幸福感爆棚！    开心

正在加载模型和分词器...


'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /hfl/chinese-roberta-wwm-ext/resolve/main/tokenizer_config.json (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x000002C2110688C0>, 'Connection to huggingface.co timed out. (connect timeout=10)'))"), '(Request ID: b7c0d9d4-d57a-4c90-982c-079415a5f3e1)')' thrown while requesting HEAD https://huggingface.co/hfl/chinese-roberta-wwm-ext/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].


In [3]:
# 保存最终模型
trainer.save_model("my_final_model/my_model_1")
tokenizer.save_pretrained("my_final_model/my_model_1")
print("模型已保存到 'my_model_1' 目录")

模型已保存到 'my_final_model' 目录
